In [8]:
import numpy as np

In [9]:
class TreeNode:
    def __init__(self, feature_idx=None, threshold=None, left_child=None, right_child=None, prediction=None,  is_leaf=False, depth=0):
        self.is_leaf = is_leaf
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left_child = left_child
        self.right_child = right_child
        self.prediction = prediction
        self.depth = depth
    
    def __repr__(self):
        return f"TreeNode(feature_idx={self.feature_idx}, threshold={self.threshold}, left_child={self.left_child}, right_child={self.right_child}, prediction={self.prediction}, is_leaf={self.is_leaf}, depth={self.depth})"
    
    def __str__(self):
        return self.__repr__()
    
root = TreeNode()
print(root)

TreeNode(feature_idx=None, threshold=None, left_child=None, right_child=None, prediction=None, is_leaf=False, depth=0)


In [32]:
class DecisionTreeClassifier():

    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, impurity_func='entropy'):
        print("running init")
        print(f"max_depth:{max_depth}, min_samples_split:{min_samples_split}, min_samples_leaf:{min_samples_leaf}, impurity_func:{impurity_func}")
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.impurity_func = impurity_func
        self.root = None

    def fit(self, X, y):
        print("running fit")
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        print("running _build_tree")
        print(f"X:{X}, y:{y}, depth:{depth}")
        n_samples = len(y)
        n_classes = len(np.unique(y))

        #stopping criterias

        if (self.max_depth is not None and depth >= self.max_depth or n_classes == 1 or n_samples <= self.min_samples_leaf):
            return TreeNode(is_leaf=True, prediction=self._majority_class(y))
        
        best_feature_idx, best_threshold, best_gain, left_mask, right_mask = self._find_best_split(X,y)

        if best_gain <= 0:
            return TreeNode(is_leaf=True, prediction=self._majority_class(y))

        left_child = self._build_tree(X[left_mask], y[left_mask], depth+1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth+1)

        return TreeNode(feature_idx = best_feature_idx, threshold=best_threshold, left_child=left_child, right_child=right_child, is_leaf=False, depth=depth)
    
    def _majority_class(self, y):
        print("running _majority_class")
        print(f"y:{y}")
        values, counts = np.unique(y, return_counts=True)
        print(f"values:{values}, counts:{counts}")
        print(f"np.argmax(counts):{np.argmax(counts)}")
        return values[np.argmax(counts)]
    
    def _find_best_split(self, X, y):
        print("running _find_best_split")
        print(f"X:{X}, y:{y}")
        best_feature = None
        best_threshold = None
        best_gain = None
        best_left_mask = None
        best_right_mask = None

        for feature_idx in range(X.shape[1]):
            feature_values = X[:, feature_idx]    
            thresholds = np.unique(feature_values)

            print(f"feature_idx:{feature_idx}, feature_values:{feature_values}, thresholds:{thresholds}")

            for threshold in thresholds:
                print(f"threshold:{threshold}")
                print(f"feature_values <= threshold:{feature_values <= threshold}")
                left_mask = feature_values <= threshold
                right_mask = ~left_mask
                print(f"left_mask:{left_mask}, right_mask:{right_mask}, left_mask.sum():{left_mask.sum()}, right_mask.sum(): {right_mask.sum()}")

                if left_mask.sum() < self.min_samples_leaf or right_mask.sum() < self.min_samples_leaf:
                    continue

                y_left = y[left_mask]   
                y_right = y[right_mask]
                print(f"y_left:{y_left}, y_right:{y_right}")

                gain = self._information_gain(y, y_left, y_right)

                if best_gain is None or gain > best_gain:
                    best_feature = feature_idx
                    best_threshold = threshold
                    best_gain = gain
                    best_left_mask = left_mask
                    best_right_mask = right_mask

        return best_feature, best_threshold, best_gain, best_left_mask, best_right_mask
    
    def _information_gain(self, y, y_left, y_right):
        print("running _information_gain")
        print(f"y:{y}, y_left:{y_left}, y_right:{y_right}")
        p = len(y_left)/len(y)
        print(f"left_p:{p}")
        return self._entropy(y) - (p*self._entropy(y_left) + (1-p)*self._entropy(y_right))
    
    def _entropy(self, y):
        print("running _entropy")
        print(f"y:{y}")
        _, counts = np.unique(y, return_counts=True)
        print(f"counts:{counts}")
        probs = counts/counts.sum()
        print(f"probs:{probs}")
        return -np.sum(probs*np.log2(probs))

    def _predict_one(self, x, node):
        print("running _predict_one")
        print(f"x:{x}, node:{node}")
        if node.is_leaf:
            return node.prediction
        if x[node.feature_idx] <= node.threshold:
            return self._predict_one(x, node.left_child)
        else:
            return self._predict_one(x, node.right_child)
        
    def predict(self, X):
        print("running predict")
        print(f"X:{X}")
        return np.array([self._predict_one(x, self.root) for x in X])


In [33]:
X = np.array([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

dtc = DecisionTreeClassifier(max_depth=5)
dtc.fit(X, y)
print(dtc.predict(X))

running init
max_depth:5, min_samples_split:2, min_samples_leaf:1, impurity_func:entropy
running fit
running _build_tree
X:[[1 2]
 [2 3]
 [3 4]
 [4 5]
 [5 6]
 [6 7]
 [7 8]
 [8 9]], y:[0 0 0 0 1 1 1 1], depth:0
running _find_best_split
X:[[1 2]
 [2 3]
 [3 4]
 [4 5]
 [5 6]
 [6 7]
 [7 8]
 [8 9]], y:[0 0 0 0 1 1 1 1]
feature_idx:0, feature_values:[1 2 3 4 5 6 7 8], thresholds:[1 2 3 4 5 6 7 8]
threshold:1
feature_values <= threshold:[ True False False False False False False False]
left_mask:[ True False False False False False False False], right_mask:[False  True  True  True  True  True  True  True], left_mask.sum():1, right_mask.sum(): 7
y_left:[0], y_right:[0 0 0 1 1 1 1]
running _information_gain
y:[0 0 0 0 1 1 1 1], y_left:[0], y_right:[0 0 0 1 1 1 1]
left_p:0.125
running _entropy
y:[0 0 0 0 1 1 1 1]
counts:[4 4]
probs:[0.5 0.5]
running _entropy
y:[0]
counts:[1]
probs:[1.]
running _entropy
y:[0 0 0 1 1 1 1]
counts:[3 4]
probs:[0.42857143 0.57142857]
threshold:2
feature_values <= thre